In [ ]:
# 01. 데이터 확인·전처리 (UCI Steel Plates Faults, 1,941건 / 측정값 27개 / 결함 7종)
import sys, warnings
sys.path.insert(0, "../src"); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from steel import *
pd.set_option("display.width", 200); pd.set_option("display.precision", 4)

In [ ]:
# 품질 점검
raw = load_raw()
pd.Series({
    "shape": raw.shape,
    "결측값": raw.isna().sum().sum(),
    "중복 행": raw.duplicated().sum(),
    "정답이 1개가 아닌 행": (raw[TARGET_COLS].sum(axis=1) != 1).sum(),
    "A300 + A400 != 1 인 행": ((raw["TypeOfSteel_A300"] + raw["TypeOfSteel_A400"]) != 1).sum(),
}, name="결과")
# → 결측·중복 없음, 결함마다 정답 1개 → 단일 target 다중 분류
# → TypeOfSteel_A400 = 1 - A300 (완전 중복) → 제거, 입력 변수 26개

In [ ]:
# 80:20 층화 분할 후 클래스 분포 (Train/Test 비율 동일)
X_train, X_test, y_train, y_test = load_data()
dist = pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()}).loc[CLASS_ORDER]
dist["train_%"] = (dist["train"] / dist["train"].sum() * 100).round(1)
print(X_train.shape, X_test.shape); display(dist)
# → 최다(Other_Faults 34.7%) vs 최소(Dirtiness 2.8%) 약 12배 → 핵심 지표 Macro F1

In [ ]:
# 왜도가 큰 변수와 극단값 (Train)
skew = X_train.skew().abs().sort_values(ascending=False).head(6)
q1, q3 = X_train["Pixels_Areas"].quantile([0.25, 0.75])
out = X_train["Pixels_Areas"] > q3 + 1.5 * (q3 - q1)
display(skew.round(2).to_frame("|skew|"))
print(f"Pixels_Areas 극단값 {out.sum()}건 중 K_Scratch 비율: {(y_train[out] == 'K_Scratch').mean():.0%}")
# → 극단값 대부분이 K_Scratch의 실제 특징(큰 면적) → 제거하지 않음, 트리 모델 위주로 진행

In [ ]:
# 결함 유형과의 상호정보량 상위 10개 (Train)
from sklearn.feature_selection import mutual_info_classif
mi = pd.Series(mutual_info_classif(X_train, y_train, random_state=RS), index=X_train.columns).sort_values(ascending=False)
mi.head(10).round(3).to_frame("MI")
# → 크기(면적·둘레)와 생산 조건(두께·폭·강종) 변수가 상위

In [ ]:
# 생산 조건(폭·두께·강종) 그룹 구조
g_tr, g_te = plate_groups(X_train), plate_groups(X_test)
top_share = pd.crosstab(g_tr, y_train).pipe(lambda t: t.max(axis=1).sum() / t.values.sum())
rng = np.random.default_rng(RS)
shuffled = np.mean([pd.crosstab(g_tr, rng.permutation(y_train.values)).pipe(lambda t: t.max(axis=1).sum() / t.values.sum()) for _ in range(100)])
pd.Series({
    "Train 생산 조건 수": g_tr.nunique(),
    "그룹 내 최다 유형 비율 (실제)": round(top_share, 3),
    "같은 계산, 정답 무작위 섞기 (100회 평균)": round(shuffled, 3),
    "Test 결함 중 Train에 같은 조건이 있는 비율": f"{g_te.isin(set(g_tr)).mean():.1%}",
}, name="값")
# → 같은 생산 조건 안에서 같은 결함이 몰려 나옴 + Test 대부분이 이미 본 조건
# → 무작위 분할 점수는 "이미 본 조건" 기준 성능 → 05에서 그룹 교차검증으로 따로 확인